# Capacitated Facility Location with NVIDIA cuOpt (MILP)

This notebook formulates and solves a capacitated facility location problem (CFLP) using the NVIDIA cuOpt MILP Python API.

## Problem Description

A logistics company operates a distribution network. They need to determine:
- Which distribution centers (DCs) to open from a set of candidate locations
- How to assign customers to open DCs to fulfill their demand

The goal is to minimize total annual logistics cost while:
- Meeting all customer demand
- Respecting DC capacity constraints
- Balancing fixed operating costs with transportation costs

### Model Assumptions

**Important**: This model makes the following simplifying assumptions:
1. **Single assignment**: Each customer is assigned to exactly one DC
2. **Capacity limits**: Each DC has a maximum pallet-handling capacity per week
3. **Fixed costs**: Opening a DC incurs a fixed annual operating cost
4. **Known demand**: Customer demand is deterministic and known in advance
5. **Euclidean distances**: Transportation costs are proportional to straight-line distance

This formulation is well-suited for:
- Distribution network design
- Warehouse location planning
- Supply chain optimization
- Retail store placement decisions


## Environment Setup

First, let's check if we have a GPU available and install necessary dependencies.


In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()


In [ ]:
# Check if cuOpt is installed, if not provide installation instructions
try:
    import cuopt
    print("✓ cuOpt is installed")
except ImportError:
    print("⚠️ cuOpt is not installed!")
    print("\nTo install cuOpt, uncomment and run ONE of the following commands:")
    print("  For CUDA 12: %pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12")
    print("  For CUDA 13: %pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13")
    print("\nThen restart the kernel and run again.")
    raise ImportError("cuOpt is required. Please install it using the instructions above.")

# Uncomment ONE of the following lines to install cuOpt (requires GPU with CUDA):
# %pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12 nvidia-nvjitlink-cu12
# %pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13 nvidia-nvjitlink-cu13


In [ ]:
%pip install matplotlib seaborn


## Import Required Libraries


In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

from cuopt.linear_programming.problem import Problem, VType, sense, LinearExpression
from cuopt.linear_programming.solver_settings import SolverSettings

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")


## Problem Statement

- We have `num_dcs` candidate distribution centers (DCs) with weekly pallet-handling capacities and fixed operating costs.
- We have `num_customers` customers, each with a weekly pallet demand and a location.
- Transportation cost is proportional to distance between DC and customer.
- We must:
  - Decide which DCs to open.
  - Assign every customer's demand to exactly one open DC.
- Objective: minimize total annualized logistics cost = fixed DC cost + transportation cost.
- Constraints:
  - Each customer's demand is fully assigned to a single DC.
  - The sum of assigned demand at each DC does not exceed that DC's capacity.
  - No customer can be assigned to a closed DC.


## Problem Data Setup

Generate a synthetic instance with candidate DC locations and customer locations.


In [ ]:
# -----------------------------
# Synthetic instance generation
# -----------------------------

rng = np.random.default_rng(42)

# Sets
num_dcs = 5
num_customers = 20

I = list(range(num_dcs))
J = list(range(num_customers))

# DC locations on a 2D grid (for visualization)
dc_coords = rng.uniform(low=0.0, high=1000.0, size=(num_dcs, 2))

# Customer locations
cust_coords = rng.uniform(low=0.0, high=1000.0, size=(num_customers, 2))

# Euclidean distances and cost per pallet
dist_matrix = np.linalg.norm(
    dc_coords[:, None, :] - cust_coords[None, :, :],
    axis=2
)

alpha = 0.05  # $ per pallet-km
unit_cost = alpha * dist_matrix

# Weekly customer demand (pallets)
demand = rng.integers(low=80, high=250, size=num_customers).astype(float)

# DC capacity: scaled so that total capacity > total demand
total_demand = demand.sum()
avg_capacity = total_demand * 1.4 / num_dcs
capacity = avg_capacity * rng.uniform(low=0.8, high=1.2, size=num_dcs)
capacity = capacity.astype(float)

# Fixed opening costs (proportional to capacity + noise)
base_fc = 80000.0
fixed_cost = base_fc + 20.0 * capacity + rng.normal(loc=0.0, scale=5000.0, size=num_dcs)
fixed_cost = np.maximum(fixed_cost, 0.0).astype(float)  # Ensure non-negative costs

# Allowed arcs: all DC–customer pairs
A = [(i, j) for i in I for j in J]

print(f"Problem Size: {num_dcs} DCs, {num_customers} customers")
print(f"Total demand: {total_demand:.0f} pallets/week")
print(f"\nDC capacities: {capacity.round(0)}")
print(f"Fixed costs: ${fixed_cost.round(0)}")


In [ ]:
# Visualize problem data
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: DC and Customer locations
ax1 = axes[0]
ax1.scatter(cust_coords[:, 0], cust_coords[:, 1], c="tab:blue", s=demand/2, alpha=0.6, label="Customers")
ax1.scatter(dc_coords[:, 0], dc_coords[:, 1], c="tab:red", marker="s", s=100, label="DCs")

for i, (xi, yi) in enumerate(dc_coords):
    ax1.text(xi + 15, yi + 15, f"DC {i}", fontsize=9, color="red")

ax1.set_xlabel("X coordinate")
ax1.set_ylabel("Y coordinate")
ax1.set_title("DC and Customer Locations\n(Customer size = demand)", fontsize=14, fontweight="bold")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: DC capacity and fixed cost
ax2 = axes[1]
x_pos = np.arange(num_dcs)
width = 0.35

bars1 = ax2.bar(x_pos - width/2, capacity, width, label='Capacity (pallets)', color='tab:blue', alpha=0.7)
ax2_twin = ax2.twinx()
bars2 = ax2_twin.bar(x_pos + width/2, fixed_cost/1000, width, label='Fixed Cost ($K)', color='tab:orange', alpha=0.7)

ax2.set_xlabel('DC')
ax2.set_ylabel('Capacity (pallets/week)', color='tab:blue')
ax2_twin.set_ylabel('Fixed Cost ($K)', color='tab:orange')
ax2.set_title('DC Capacity and Fixed Costs', fontsize=14, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'DC {i}' for i in I])
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()


## Problem Formulation

### Decision Variables
- `y[i]`: Binary, 1 if DC i is open, 0 otherwise
- `x[i,j]`: Binary, 1 if customer j is assigned to DC i, 0 otherwise

### Objective Function
Minimize: Fixed DC costs + Transportation costs

$$\min \sum_i f_i y_i + \sum_{i,j} c_{ij} d_j x_{ij}$$

### Constraints
1. **Assignment**: Each customer assigned to exactly one DC: $\sum_i x_{ij} = 1$ for each customer $j$
2. **Capacity/Linking**: DC load cannot exceed capacity: $\sum_j d_j x_{ij} \le u_i y_i$ for each DC $i$
3. **Binary**: $x_{ij} \in \{0,1\}$, $y_i \in \{0,1\}$


In [ ]:
# -----------------------------
# Problem and variable creation
# -----------------------------

prob = Problem("CFLP")

# Decision variables
y = {}  # DC open/close binaries
x = {}  # assignment binaries

# y_i: 1 if DC i is open
for i in I:
    y[i] = prob.addVariable(
        name=f"y_{i}",
        lb=0.0,
        ub=1.0,
        vtype=VType.INTEGER
    )

# x_ij: 1 if customer j is assigned to DC i
for (i, j) in A:
    x[i, j] = prob.addVariable(
        name=f"x_{i}_{j}",
        lb=0.0,
        ub=1.0,
        vtype=VType.INTEGER
    )

print(f"Number of variables: {prob.NumVariables}")
print(f"  - DC open/close (y): {num_dcs}")
print(f"  - Assignment (x): {num_dcs * num_customers}")


In [ ]:
# -----------------------------
# Objective function
# -----------------------------

obj = LinearExpression([], [], 0.0)

# Fixed facility costs
for i in I:
    obj += float(fixed_cost[i]) * y[i]

# Transportation costs: c_ij * d_j * x_ij
for (i, j) in A:
    flow_cost = float(unit_cost[i, j] * demand[j])
    obj += flow_cost * x[i, j]

prob.setObjective(obj, sense=sense.MINIMIZE)

print("Objective set: Minimize (Fixed DC Costs + Transportation Costs)")


In [ ]:
# -----------------------------
# Constraints
# -----------------------------

# 1) Each customer assigned to exactly one DC
for j in J:
    expr = LinearExpression([], [], 0.0)
    for i in I:
        expr += x[i, j]
    prob.addConstraint(expr == 1.0, name=f"assign_{j}")

print(f"Assignment constraints added: {len(J)}")

# 2) DC capacity and logical linking: sum_j d_j x_ij <= capacity[i] * y_i
for i in I:
    expr = LinearExpression([], [], 0.0)
    for j in J:
        expr += float(demand[j]) * x[i, j]
    expr -= float(capacity[i]) * y[i]
    prob.addConstraint(expr <= 0.0, name=f"capacity_{i}")

print(f"Capacity constraints added: {len(I)}")
print(f"\nTotal constraints: {prob.NumConstraints}")


## Solve the MILP with cuOpt

We now configure a few basic MILP parameters and call `solve()`.


In [ ]:
# -----------------------------
# Solve
# -----------------------------

settings = SolverSettings()

# Example tuning parameters (adjust according to your environment)
settings.set_parameter("time_limit", "300.0")          # seconds
settings.set_parameter("mip_relative_gap", "1e-4")     # relative optimality gap

print("Starting optimization...")
print(f"Problem size: {prob.NumVariables} variables, {prob.NumConstraints} constraints")
print()

start_time = time.time()
prob.solve(settings=settings)
solve_time = time.time() - start_time

print(f"\nOptimization completed in {solve_time:.3f} seconds")
print(f"Solve status: {prob.Status}")
print(f"Best objective value: ${prob.ObjValue:,.2f}")
print(f"Solve time (internal): {prob.SolveTime:.3f}s")


## Extract and Analyze the Solution

We inspect which DCs are opened, how customers are assigned, and the resulting loads on each DC.


In [ ]:
# -----------------------------
# Solution extraction
# -----------------------------

# Status codes: 1 = Optimal, 2 = Feasible, others indicate issues
# See cuOpt documentation for full status code reference
if prob.Status not in (1, 2):
    status_msg = {0: "Unknown", 3: "Infeasible", 4: "Unbounded", 5: "Time Limit"}.get(prob.Status, "Unknown")
    raise RuntimeError(f"Solver did not find a solution: Status = {prob.Status} ({status_msg})")

# Open DCs
open_dcs = [i for i in I if y[i].getValue() > 0.5]
print("Open DCs:", open_dcs)

# Assignment matrix
assign_matrix = np.zeros((num_dcs, num_customers))
for (i, j) in A:
    val = x[i, j].getValue()
    assign_matrix[i, j] = val

# For each customer j, identify assigned DC i
assigned_dc = assign_matrix.argmax(axis=0)

# Load per DC
dc_load = assign_matrix @ demand

print(f"\nDC loads: {dc_load.round(0)}")


In [ ]:
# Create summary DataFrames
dc_df = pd.DataFrame({
    "DC": I,
    "Open": ["Yes" if i in open_dcs else "No" for i in I],
    "Capacity": capacity.round(0),
    "Load": dc_load.round(0),
    "Utilization %": (dc_load / capacity * 100).round(1),
    "Fixed Cost ($)": fixed_cost.round(0)
})

cust_df = pd.DataFrame({
    "Customer": J,
    "Demand": demand.round(0),
    "Assigned DC": assigned_dc,
    "Transport Cost ($)": [unit_cost[assigned_dc[j], j] * demand[j] for j in J]
})

print("\n" + "="*70)
print(" " * 20 + "DC SUMMARY")
print("="*70)
display(dc_df)

print("\n" + "="*70)
print(" " * 20 + "CUSTOMER ASSIGNMENTS")
print("="*70)
display(cust_df)


In [ ]:
# Calculate cost breakdown
total_fixed_cost = sum(fixed_cost[i] for i in open_dcs)
total_transport_cost = sum(unit_cost[assigned_dc[j], j] * demand[j] for j in J)

print("\n" + "="*70)
print(" " * 20 + "COST BREAKDOWN")
print("="*70)
print(f"\n📊 Cost Summary:")
print(f"   • Total Fixed Costs: ${total_fixed_cost:,.2f}")
print(f"   • Total Transport Costs: ${total_transport_cost:,.2f}")
print(f"   • Total Logistics Cost: ${prob.ObjValue:,.2f}")
if prob.ObjValue > 0:
    print(f"   • Fixed Cost %: {total_fixed_cost/prob.ObjValue*100:.1f}%")
    print(f"   • Transport Cost %: {total_transport_cost/prob.ObjValue*100:.1f}%")

print(f"\n🏭 Network Design:")
print(f"   • DCs Opened: {len(open_dcs)} of {num_dcs}")
print(f"   • Customers Served: {num_customers}")
print(f"   • Total Demand Met: {total_demand:,.0f} pallets/week")

print(f"\n📦 Utilization:")
if open_dcs:
    avg_util = np.mean([dc_load[i]/capacity[i]*100 for i in open_dcs])
    print(f"   • Average DC Utilization: {avg_util:.1f}%")
    for i in open_dcs:
        print(f"   • DC {i}: {dc_load[i]/capacity[i]*100:.1f}% ({dc_load[i]:.0f}/{capacity[i]:.0f})")
else:
    print("   • No DCs opened (unexpected - check solver status)")


## Visualize the Network Design

We now plot DCs, customers, and assignment arcs in a 2D plane (using the synthetic coordinates defined above).


In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Network design with assignments
ax1 = axes[0, 0]

# Draw assignment lines
for j in J:
    i = assigned_dc[j]
    x1, y1 = dc_coords[i]
    x2, y2 = cust_coords[j]
    ax1.plot([x1, x2], [y1, y2],
             color="tab:green" if i in open_dcs else "lightgray",
             linewidth=0.7, alpha=0.7)

# Plot customers
ax1.scatter(cust_coords[:, 0], cust_coords[:, 1], c="tab:blue", s=demand/2, alpha=0.6, label="Customers")
for j, (xj, yj) in enumerate(cust_coords):
    ax1.text(xj + 5, yj + 5, str(j), fontsize=8, color="blue")

# Plot DCs
colors = ["tab:red" if i in open_dcs else "gray" for i in I]
ax1.scatter(dc_coords[:, 0], dc_coords[:, 1], c=colors, marker="s", s=80, label="DCs")

for i, (xi, yi) in enumerate(dc_coords):
    label = f"DC {i}"
    if i in open_dcs:
        label += " ✓"
    ax1.text(xi + 5, yi + 5, label, fontsize=8, color="red" if i in open_dcs else "gray")

ax1.set_xlabel("X coordinate")
ax1.set_ylabel("Y coordinate")
ax1.set_title("CFLP Network Design Solution", fontsize=14, fontweight="bold")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: DC utilization
ax2 = axes[0, 1]
dc_colors = ['tab:green' if i in open_dcs else 'lightgray' for i in I]
bars = ax2.bar([f'DC {i}' for i in I], dc_load, color=dc_colors, alpha=0.7, label='Load')
ax2.bar([f'DC {i}' for i in I], capacity - dc_load, bottom=dc_load, color='white', edgecolor='gray', alpha=0.5, label='Unused')

# Add capacity line markers
for i, bar in enumerate(bars):
    ax2.hlines(capacity[i], bar.get_x(), bar.get_x() + bar.get_width(), colors='red', linestyles='--', linewidth=2)

ax2.set_ylabel('Pallets/week')
ax2.set_title('DC Load vs Capacity', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Cost breakdown pie chart
ax3 = axes[1, 0]
cost_labels = ['Fixed Costs', 'Transport Costs']
cost_values = [total_fixed_cost, total_transport_cost]
colors_pie = ['tab:red', 'tab:blue']
ax3.pie(cost_values, labels=cost_labels, autopct='%1.1f%%', colors=colors_pie, startangle=90)
ax3.set_title('Cost Breakdown', fontsize=14, fontweight='bold')

# Plot 4: Customers per DC
ax4 = axes[1, 1]
customers_per_dc = [sum(1 for j in J if assigned_dc[j] == i) for i in I]
demand_per_dc = dc_load

x_pos = np.arange(num_dcs)
width = 0.35
bars1 = ax4.bar(x_pos - width/2, customers_per_dc, width, label='# Customers', color='tab:blue', alpha=0.7)
ax4_twin = ax4.twinx()
bars2 = ax4_twin.bar(x_pos + width/2, demand_per_dc, width, label='Demand (pallets)', color='tab:orange', alpha=0.7)

ax4.set_xlabel('DC')
ax4.set_ylabel('Number of Customers', color='tab:blue')
ax4_twin.set_ylabel('Demand (pallets)', color='tab:orange')
ax4.set_title('Customer and Demand Distribution', fontsize=14, fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([f'DC {i}' for i in I])
ax4.legend(loc='upper left')
ax4_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()


## Extension: Semi-Relaxation (Fractional Customer Assignments)

The following section demonstrates a **semi-relaxation** of the MILP:
- **DC open/close decisions (y)**: Remain INTEGER (binary)
- **Customer assignment (x)**: Changed to CONTINUOUS (fractional allowed)

This allows customers to be partially assigned to multiple DCs, which can provide a lower bound on the optimal MILP solution or be used when fractional assignments are acceptable (e.g., splitting orders across DCs).


In [ ]:
# -----------------------------
# Semi-Relaxation: Fractional assignments allowed
# -----------------------------

# Create a new problem for the semi-relaxed version
prob_lp = Problem("CFLP_SemiRelaxed")

# Decision variables
y_lp = {}  # DC open/close - remain INTEGER (binary)
x_lp = {}  # assignment - now CONTINUOUS (fractional allowed)

# y_i: 1 if DC i is open (still binary/integer - facility decisions remain discrete)
for i in I:
    y_lp[i] = prob_lp.addVariable(
        name=f"y_lp_{i}",
        lb=0.0,
        ub=1.0,
        vtype=VType.INTEGER  # Still INTEGER (DC open/close decision)
    )

# x_ij: Fractional assignment (0 <= x_ij <= 1), now CONTINUOUS
for (i, j) in A:
    x_lp[i, j] = prob_lp.addVariable(
        name=f"x_lp_{i}_{j}",
        lb=0.0,
        ub=1.0,
        vtype=VType.CONTINUOUS  # Changed to CONTINUOUS for fractional assignments
    )

print("Semi-Relaxed Problem - Number of variables:", prob_lp.NumVariables)
print("Note: y_i remain INTEGER (facility decisions), x_ij are CONTINUOUS (fractional assignments)")


In [ ]:
# -----------------------------
# Objective function for relaxed LP
# -----------------------------

obj_lp = LinearExpression([], [], 0.0)

# Fixed facility costs (same as MILP)
for i in I:
    obj_lp += float(fixed_cost[i]) * y_lp[i]

# Transportation costs: c_ij * d_j * x_ij (same as MILP)
for (i, j) in A:
    flow_cost = float(unit_cost[i, j] * demand[j])
    obj_lp += flow_cost * x_lp[i, j]

prob_lp.setObjective(obj_lp, sense=sense.MINIMIZE)

print("Semi-relaxed objective set (minimize total cost).")


In [ ]:
# -----------------------------
# Constraints for relaxed LP
# -----------------------------

# 1) Each customer's demand must be fully assigned (sum of fractional assignments = 1)
for j in J:
    expr_lp = LinearExpression([], [], 0.0)
    for i in I:
        expr_lp += x_lp[i, j]
    prob_lp.addConstraint(expr_lp == 1.0, name=f"assign_lp_{j}")

print(f"Assignment constraints added (fractional allowed): {len(J)}")

# 2) DC capacity and logical linking: sum_j d_j x_ij <= capacity[i] * y_i
for i in I:
    expr_lp = LinearExpression([], [], 0.0)
    for j in J:
        expr_lp += float(demand[j]) * x_lp[i, j]
    expr_lp -= float(capacity[i]) * y_lp[i]
    prob_lp.addConstraint(expr_lp <= 0.0, name=f"capacity_lp_{i}")

print(f"Capacity constraints added: {len(I)}")


In [ ]:
# -----------------------------
# Solve the semi-relaxed problem
# -----------------------------

settings_lp = SolverSettings()
settings_lp.set_parameter("time_limit", "300.0")

prob_lp.solve(settings=settings_lp)

print(f"Semi-Relaxed Solve status: {prob_lp.Status}")
print(f"Semi-Relaxed Objective value: ${prob_lp.ObjValue:,.2f}")
print(f"Solve time: {prob_lp.SolveTime:.3f}s")

# Compare with MILP solution (only if both solved successfully)
if prob.Status in (1, 2) and prob_lp.Status in (1, 2):
    gap = prob.ObjValue - prob_lp.ObjValue
    print(f"\nComparison:")
    print(f"  MILP (full integer) objective: ${prob.ObjValue:,.2f}")
    print(f"  Semi-relaxed objective:        ${prob_lp.ObjValue:,.2f}")
    if prob.ObjValue > 0:
        print(f"  Integrality gap: ${gap:,.2f} ({gap/prob.ObjValue*100:.2f}%)")
    else:
        print(f"  Integrality gap: ${gap:,.2f}")


In [ ]:
# -----------------------------
# Analyze fractional assignment solution
# -----------------------------

if prob_lp.Status not in (1, 2):
    status_msg = {0: "Unknown", 3: "Infeasible", 4: "Unbounded", 5: "Time Limit"}.get(prob_lp.Status, "Unknown")
    raise RuntimeError(f"Semi-relaxed solver did not find a solution: Status = {prob_lp.Status} ({status_msg})")

# Open DCs (still integer)
open_dcs_lp = [i for i in I if y_lp[i].getValue() > 0.5]
print("Open DCs (relaxed LP):", open_dcs_lp)

# Fractional assignment matrix
assign_matrix_lp = np.zeros((num_dcs, num_customers))
for (i, j) in A:
    val = x_lp[i, j].getValue()
    assign_matrix_lp[i, j] = val

# For each customer, show fractional assignments
print("\nFractional assignments (customer -> DC: fraction):")
for j in J:
    assignments = [(i, assign_matrix_lp[i, j]) for i in I if assign_matrix_lp[i, j] > 1e-6]
    if len(assignments) > 1:  # Only show customers with split assignments
        assignment_str = ", ".join([f"DC{i}: {frac:.3f}" for i, frac in assignments])
        print(f"  Customer {j}: {assignment_str}")

# Load per DC
dc_load_lp = assign_matrix_lp @ demand
print(f"\nDC loads (relaxed LP): {dc_load_lp.round(0)}")


## Extension: Multi-Period Capacity Expansion

This section extends the CFLP to a multi-period setting where we decide **when** to open DCs (now vs defer) and handle time-varying demand. Key features:

- **Multiple time periods**: Plan over T periods (e.g., quarters or years)
- **Deferred opening decisions**: DCs can be opened in any period, not just period 0
- **Time-discounted costs**: Future costs are discounted using a discount factor
- **Time-varying demand**: Customer demand may change over periods
- **Capacity constraints**: DCs can only serve customers after they're opened
- **Objective**: Minimize total discounted cost over all periods


In [ ]:
# -----------------------------
# Multi-period problem setup
# -----------------------------

# Number of time periods
num_periods = 3  # e.g., periods 0, 1, 2 (now, next year, year after)
T = list(range(num_periods))

# Discount factor (e.g., 0.95 means $1 next period is worth $0.95 today)
discount_factor = 0.95

# Time-varying demand: demand may grow over time
demand_growth_rate = 1.1  # 10% growth per period
demand_period = {}
for t in T:
    demand_period[t] = demand * (demand_growth_rate ** t)

# Time-varying fixed costs: opening costs may change over time
cost_inflation = 1.05  # 5% cost increase per period
fixed_cost_period = {}
for t in T:
    fixed_cost_period[t] = fixed_cost * (cost_inflation ** t)

# Transportation costs may also vary over time
unit_cost_period = {}
for t in T:
    unit_cost_period[t] = unit_cost * (1.0 + 0.02 * t)  # 2% increase per period

print(f"Multi-period setup:")
print(f"  Number of periods: {num_periods}")
print(f"  Discount factor: {discount_factor}")
print(f"  Demand growth rate: {(demand_growth_rate - 1) * 100:.1f}% per period")
print(f"  Cost inflation: {(cost_inflation - 1) * 100:.1f}% per period")
print(f"\nTotal demand by period:")
for t in T:
    total_demand_t = demand_period[t].sum()
    print(f"  Period {t}: {total_demand_t:,.1f} pallets")


In [ ]:
# -----------------------------
# Multi-period problem and variable creation
# -----------------------------

prob_mp = Problem("CFLP_MultiPeriod")

# Decision variables
y_mp = {}  # y_mp[i][t]: 1 if DC i is open at the START of period t (state variable)
z_mp = {}  # z_mp[i][t]: 1 if DC i opens specifically in period t (decision variable)
x_mp = {}  # x_mp[i][j][t]: 1 if customer j is assigned to DC i in period t

# y_mp[i][t]: State variable tracking if DC i is open at start of period t
for i in I:
    y_mp[i] = {}
    for t in T:
        y_mp[i][t] = prob_mp.addVariable(
            name=f"y_{i}_{t}",
            lb=0.0,
            ub=1.0,
            vtype=VType.INTEGER
        )

# z_mp[i][t]: 1 if DC i opens specifically in period t
for i in I:
    z_mp[i] = {}
    for t in T:
        z_mp[i][t] = prob_mp.addVariable(
            name=f"z_{i}_{t}",
            lb=0.0,
            ub=1.0,
            vtype=VType.INTEGER
        )

# x_mp[i][j][t]: assignment in period t
for i in I:
    x_mp[i] = {}
    for j in J:
        x_mp[i][j] = {}
        for t in T:
            x_mp[i][j][t] = prob_mp.addVariable(
                name=f"x_{i}_{j}_{t}",
                lb=0.0,
                ub=1.0,
                vtype=VType.INTEGER
            )

print(f"Multi-period - Number of variables: {prob_mp.NumVariables}")
print(f"  DC opening state variables (y): {num_dcs * num_periods}")
print(f"  DC opening decision variables (z): {num_dcs * num_periods}")
print(f"  Assignment variables (x): {num_dcs * num_customers * num_periods}")


In [ ]:
# -----------------------------
# Multi-period constraints
# -----------------------------

# 1) DC opening logic constraints
for i in I:
    # Period 0: y_mp[i][0] = z_mp[i][0]
    expr = LinearExpression([], [], 0.0)
    expr += y_mp[i][0]
    expr -= z_mp[i][0]
    prob_mp.addConstraint(expr == 0.0, name=f"open_logic_{i}_0")
    
    # Period t > 0: State transition constraints
    for t in T[1:]:
        # Monotonicity: once open, stays open
        expr = LinearExpression([], [], 0.0)
        expr += y_mp[i][t]
        expr -= y_mp[i][t-1]
        prob_mp.addConstraint(expr >= 0.0, name=f"monotone_{i}_{t}")
        
        # If opens in period t, must be open
        expr = LinearExpression([], [], 0.0)
        expr += y_mp[i][t]
        expr -= z_mp[i][t]
        prob_mp.addConstraint(expr >= 0.0, name=f"open_if_opens_{i}_{t}")
        
        # Can only be open if was open before OR opens now
        expr = LinearExpression([], [], 0.0)
        expr += y_mp[i][t]
        expr -= y_mp[i][t-1]
        expr -= z_mp[i][t]
        prob_mp.addConstraint(expr <= 0.0, name=f"open_logic_{i}_{t}")

# 2) Each DC can open at most once
for i in I:
    expr = LinearExpression([], [], 0.0)
    for t in T:
        expr += z_mp[i][t]
    prob_mp.addConstraint(expr <= 1.0, name=f"open_once_{i}")

print("DC opening logic constraints added")

# 3) Assignment constraints: Each customer fully assigned in each period
for j in J:
    for t in T:
        expr = LinearExpression([], [], 0.0)
        for i in I:
            expr += x_mp[i][j][t]
        prob_mp.addConstraint(expr == 1.0, name=f"assign_{j}_{t}")

print(f"Assignment constraints added: {num_customers * num_periods}")

# 4) Capacity constraints
for i in I:
    for t in T:
        expr = LinearExpression([], [], 0.0)
        demand_t = demand_period[t]
        for j in J:
            d_jt = demand_t[j]
            expr += float(d_jt) * x_mp[i][j][t]
        expr -= float(capacity[i]) * y_mp[i][t]
        prob_mp.addConstraint(expr <= 0.0, name=f"capacity_{i}_{t}")

print(f"Capacity constraints added: {num_dcs * num_periods}")
print(f"\nTotal constraints: {prob_mp.NumConstraints}")


In [ ]:
# -----------------------------
# Multi-period objective function
# -----------------------------

obj_mp = LinearExpression([], [], 0.0)

# Fixed opening costs (discounted)
for i in I:
    for t in T:
        discount = discount_factor ** t
        cost_t = fixed_cost_period[t][i]
        obj_mp += float(cost_t * discount) * z_mp[i][t]

# Transportation costs (discounted)
for i in I:
    for j in J:
        for t in T:
            discount = discount_factor ** t
            demand_t = demand_period[t]
            d_jt = demand_t[j]
            cost_t = unit_cost_period[t][i, j]
            flow_cost = float(cost_t * d_jt * discount)
            obj_mp += flow_cost * x_mp[i][j][t]

prob_mp.setObjective(obj_mp, sense=sense.MINIMIZE)

print("Multi-period objective set (minimize total discounted cost).")


In [ ]:
# -----------------------------
# Solve multi-period problem
# -----------------------------

settings_mp = SolverSettings()
settings_mp.set_parameter("time_limit", "600.0")  # Longer time limit
settings_mp.set_parameter("mip_relative_gap", "1e-4")

print("Starting multi-period optimization...")
print(f"Problem size: {prob_mp.NumVariables} variables, {prob_mp.NumConstraints} constraints")
print()

start_time = time.time()
prob_mp.solve(settings=settings_mp)
solve_time_mp = time.time() - start_time

print(f"\nOptimization completed in {solve_time_mp:.3f} seconds")
print(f"Multi-period Solve status: {prob_mp.Status}")
print(f"Multi-period Objective value (discounted): ${prob_mp.ObjValue:,.2f}")
print(f"Solve time (internal): {prob_mp.SolveTime:.3f}s")


In [ ]:
# -----------------------------
# Analyze multi-period solution
# -----------------------------

if prob_mp.Status not in (1, 2):
    status_msg = {0: "Unknown", 3: "Infeasible", 4: "Unbounded", 5: "Time Limit"}.get(prob_mp.Status, "Unknown")
    raise RuntimeError(f"Multi-period solver did not find a solution: Status = {prob_mp.Status} ({status_msg})")

# Extract opening decisions by period
opening_schedule = {}
dc_open_period = {}

for t in T:
    opening_schedule[t] = []
    for i in I:
        if z_mp[i][t].getValue() > 0.5:
            opening_schedule[t].append(i)
            dc_open_period[i] = t

never_open = [i for i in I if i not in dc_open_period]

print("DC Opening Schedule:")
for t in T:
    if opening_schedule[t]:
        print(f"  Period {t}: Open DCs {opening_schedule[t]}")
    else:
        print(f"  Period {t}: No DCs opened")
if never_open:
    print(f"  Never opened: DCs {never_open}")


In [ ]:
# Calculate costs by period
costs_by_period = {}
for t in T:
    opening_cost = 0.0
    transport_cost = 0.0
    
    # Opening costs
    for i in opening_schedule[t]:
        cost_t = fixed_cost_period[t][i]
        opening_cost += cost_t
    
    # Transportation costs
    for i in I:
        for j in J:
            if x_mp[i][j][t].getValue() > 0.5:
                demand_t = demand_period[t]
                d_jt = demand_t[j]
                cost_t = unit_cost_period[t][i, j]
                transport_cost += cost_t * d_jt
    
    discount = discount_factor ** t
    costs_by_period[t] = {
        'opening_cost': opening_cost,
        'transport_cost': transport_cost,
        'total_cost': opening_cost + transport_cost,
        'discounted_cost': (opening_cost + transport_cost) * discount
    }

# Create summary DataFrame
summary_data = []
for t in T:
    summary_data.append({
        'Period': t,
        'DCs Opened': len(opening_schedule[t]),
        'Opening Cost ($)': costs_by_period[t]['opening_cost'],
        'Transport Cost ($)': costs_by_period[t]['transport_cost'],
        'Total Cost ($)': costs_by_period[t]['total_cost'],
        'Discounted Cost ($)': costs_by_period[t]['discounted_cost'],
        'Total Demand': demand_period[t].sum()
    })

summary_df = pd.DataFrame(summary_data)
print("\nMulti-Period Cost Summary:")
display(summary_df.round(2))

print(f"\nTotal discounted cost: ${prob_mp.ObjValue:,.2f}")
print(f"Total undiscounted cost: ${summary_df['Total Cost ($)'].sum():,.2f}")


In [ ]:
# -----------------------------
# Visualize multi-period solution
# -----------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Opening schedule timeline
ax1 = axes[0]
for i in I:
    if i in dc_open_period:
        t_open = dc_open_period[i]
        ax1.barh(i, num_periods - t_open, left=t_open, alpha=0.6, color='tab:green')
        ax1.text(t_open + 0.1, i, f'DC {i} opens', va='center', fontsize=9)
    else:
        ax1.barh(i, 0.1, left=0, alpha=0.3, color='gray')
        ax1.text(0.15, i, f'DC {i} (never opened)', va='center', fontsize=9, color='gray')

ax1.set_xlabel('Period')
ax1.set_ylabel('DC')
ax1.set_title('DC Opening Schedule', fontsize=14, fontweight='bold')
ax1.set_yticks(I)
ax1.set_yticklabels([f'DC {i}' for i in I])
ax1.set_xticks(T)
ax1.set_xlim(-0.2, num_periods)
ax1.grid(True, alpha=0.3)

# Plot 2: Costs by period
ax2 = axes[1]
periods = summary_df['Period']
width = 0.35
x = np.arange(len(periods))

ax2.bar(x - width/2, summary_df['Opening Cost ($)'], width, label='Opening Cost', alpha=0.8, color='tab:red')
ax2.bar(x + width/2, summary_df['Transport Cost ($)'], width, label='Transport Cost', alpha=0.8, color='tab:blue')

ax2.set_xlabel('Period')
ax2.set_ylabel('Cost ($)')
ax2.set_title('Costs by Period (Undiscounted)', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f'Period {t}' for t in periods])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## Summary and Business Insights


In [ ]:
print("\n" + "="*70)
print(" " * 20 + "BUSINESS INSIGHTS")
print("="*70 + "\n")

print("📊 Single-Period MILP Results:")
print(f"   • Optimal Cost: ${prob.ObjValue:,.2f}")
print(f"   • DCs Opened: {len(open_dcs)} of {num_dcs}")
print(f"   • Solve Time: {solve_time:.3f} seconds")
print()

print("📈 Multi-Period Results:")
print(f"   • Total Discounted Cost: ${prob_mp.ObjValue:,.2f}")
print(f"   • Planning Horizon: {num_periods} periods")
print(f"   • DCs Eventually Opened: {len(dc_open_period)} of {num_dcs}")
print(f"   • Solve Time: {solve_time_mp:.3f} seconds")
print()

print("🔄 Semi-Relaxation Insights:")
if prob_lp.Status in (1, 2) and prob.Status in (1, 2):
    gap = prob.ObjValue - prob_lp.ObjValue
    print(f"   • Semi-Relaxed Objective: ${prob_lp.ObjValue:,.2f}")
    if prob.ObjValue > 0:
        print(f"   • Integrality Gap: ${gap:,.2f} ({gap/prob.ObjValue*100:.2f}%)")
    else:
        print(f"   • Integrality Gap: ${gap:,.2f}")
else:
    print("   • Solution not available")
print()

print("⚡ Performance:")
print(f"   • GPU Acceleration: Enabled ✓")
print(f"   • Single-period problem: {prob.NumVariables} variables, {prob.NumConstraints} constraints")
print(f"   • Multi-period problem: {prob_mp.NumVariables} variables, {prob_mp.NumConstraints} constraints")
print("\n" + "="*70)


## Conclusion

This notebook demonstrated how to use the cuOpt linear programming API to solve a Capacitated Facility Location Problem (CFLP).

### What We Achieved:
1. **Optimal DC selection** to minimize total logistics cost
2. **Customer-to-DC assignment** respecting capacity constraints
3. **LP relaxation** for lower bound analysis
4. **Multi-period extension** for capacity expansion planning

### Key Advantages of cuOpt:
- **GPU Acceleration**: Significantly faster than CPU-based solvers
- **Scalability**: Handles problems with thousands of variables and constraints
- **Integration**: Easy to integrate into existing Python workflows
- **Real-time**: Fast enough for operational decision-making

### Possible Extensions:

**High Priority - Service Levels:**
- Maximum distance constraints (customers must be within X km of assigned DC)
- Multiple service tiers with different SLAs
- Backup DC assignments for redundancy

**Additional Enhancements:**
- Multiple product types with different handling requirements
- DC capacity expansion options (modular capacity)
- Stochastic demand scenarios
- Multi-echelon supply chain (plants → DCs → customers)
- Inventory considerations and safety stock
- Seasonal demand patterns


SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
